In [ ]:
from signal import default_int_handler, signal

import matplotlib.pyplot as plt
from IPython.display import clear_output
from nn import *


class Conv(Conv1d):
    def forward(s, x):
        return super().forward(x.mT).mT[:, : x.size(1) // s.stride[0]]


class ConvTranspose(ConvTranspose1d):
    def forward(s, x):
        return functional.pad(super().forward(x.mT).mT[:, : x.size(1) * s.stride[0] - 1], (0, 0, 1, 0))


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.mT).mT


def Log(*A, N=100):
    X = n = 0
    a = []
    A = np.empty((0,) + A, "f")

    def Log(x=None):
        nonlocal X, n, A
        if x is None:
            if a:
                A = np.concatenate((A, np.stack(a) / N))
                a.clear()
            return A
        X += x
        n += 1
        if n >= N:
            a.append(X.to("cpu", torch.float, True).numpy())
            X = n = 0

    return Log


def Lin(i, o=0, b=False):
    return Linear(i, o or i, b)


def Ba(i):
    return Seq(
        BatchNorm(i, track_running_stats=False),
        SiLU(True),
    )


def Cov(i, n):
    return Seq(
        *[
            Seq(
                Conv(i, i, 3, padding=2, bias=False),
                Ba(i),
            )
            for n in range(n)
        ]
    )


def Covs(i, *n, before=lambda: 0, after=lambda b: 0):
    return Seq(
        *[
            Add(
                Seq(
                    before(),
                    Cov(i, n),
                    after(False),
                ),
                before() or after(True) or Lin(i),
            )
            for n in n
        ]
    )


def Covd(i, *n):
    return Seq(
        Covs(i, *n[:-1]),
        Add(
            Seq(
                Cov(i, n[-1]),
                Conv(i, i, 3, 2, 1, bias=False),
                Ba(i),
            ),
            Conv(i, i, 3, 2, 1, bias=False),
        ),
    )


def Covu(i, *n):
    return Seq(
        Add(
            Seq(
                ConvTranspose(i, i, 3, 2, bias=False),
                Ba(i),
                Cov(i, n[0]),
            ),
            ConvTranspose(i, i, 3, 2, bias=False),
        ),
        Covs(i, *n[1:]),
    )


def Ret(i, h=0, k=0, v=0, **_):
    h = h or (i / 4).__ceil__()
    k = k or (i / h).__ceil__()
    v = v or k
    return Add(
        Seq(
            Add(
                Seq(
                    Dot(
                        Seq(Lin(i, h * k, True), Unflatten(-1, (h, 1, k))),
                        Seq(
                            Mul(
                                Seq(Lin(i, h * k, True), Unflatten(-1, (h, 1, k))),
                                Seq(Lin(i, h * v, True), Unflatten(-1, (h, v, 1))),
                            ),
                            Decay((h, 1, k), (h, v, 1), d=-4, shift=1, **_),
                        ),
                    ),
                    Flatten(-3),
                    Lin(h * v, i),
                ),
                Lin(i, i),
            ),
            Ba(i),
        ),
        Lin(i, i),
    )


dat = Dat("../dat/py.txt", 0.01, 1024, 129)

In [ ]:
m, r, c = 64, [16], [3]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, *r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, *r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, *r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, *r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, *r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, *r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, *r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, [16], [3]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, *r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, *r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, *r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, *r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, *r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, [16], [2]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, *r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, *r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, *r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, *r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, *r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, *r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, *r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, [16], [2]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, *r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, *r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, *r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, *r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, *r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, [16], [1]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, *r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, *r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, *r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, *r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, *r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, *r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, *r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, [16], [1]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, *r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, *r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, *r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, *r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, *r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, {"h": 64}, [4]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, **r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, **r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, **r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, **r),
                                    Covs(m, *c),
                                    Add(
                                        Seq(
                                            Covd(m, *c),
                                            Ret(m, **r),
                                            Covu(m, *c),
                                        ),
                                        Lin(m),
                                    ),
                                    Covs(m, *c),
                                    Ret(m, **r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, **r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, **r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, **r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, {"h": 64}, [5]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, **r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, **r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, **r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, **r),
                                    Covs(m, *c),
                                    Add(
                                        Seq(
                                            Covd(m, *c),
                                            Ret(m, **r),
                                            Covu(m, *c),
                                        ),
                                        Lin(m),
                                    ),
                                    Covs(m, *c),
                                    Ret(m, **r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, **r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, **r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, **r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, {"h": 64, "min": 4, "max": 32}, [3]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, **r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, **r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, **r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, **r),
                                    Covs(m, *c),
                                    Add(
                                        Seq(
                                            Covd(m, *c),
                                            Ret(m, **r),
                                            Covu(m, *c),
                                        ),
                                        Lin(m),
                                    ),
                                    Covs(m, *c),
                                    Ret(m, **r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, **r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, **r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(1)
    ],
    Ret(m, **r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
m, r, c = 64, {"h": 64, "min": 4, "max": 16}, [2]
m = Seq(
    Covs(m, *c, before=lambda: Embedding(68, m, scale_grad_by_freq=True)),
    *[
        Seq(
            Ret(m, **r),
            Covs(m, *c),
            Add(
                Seq(
                    Covd(m, *c),
                    Ret(m, **r),
                    Covs(m, *c),
                    Add(
                        Seq(
                            Covd(m, *c),
                            Ret(m, **r),
                            Covs(m, *c),
                            Add(
                                Seq(
                                    Covd(m, *c),
                                    Ret(m, **r),
                                    Covs(m, *c),
                                    Add(
                                        Seq(
                                            Covd(m, *c),
                                            Ret(m, **r),
                                            Covu(m, *c),
                                        ),
                                        Lin(m),
                                    ),
                                    Covs(m, *c),
                                    Ret(m, **r),
                                    Covu(m, *c),
                                ),
                                Lin(m),
                            ),
                            Covs(m, *c),
                            Ret(m, **r),
                            Covu(m, *c),
                        ),
                        Lin(m),
                    ),
                    Covs(m, *c),
                    Ret(m, **r),
                    Covu(m, *c),
                ),
                Lin(m),
            ),
            Covs(m, *c),
        )
        for _ in range(2)
    ],
    Ret(m, **r),
    Covs(m, *c, after=lambda b: Lin(m, 68, b)),
)
print(m)
m = torch.compile(torch.jit.trace(m.to("cuda", non_blocking=True), dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-3)
s = torch.cuda.amp.GradScaler()
L = 9e9
logl = Log()
loga = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100 or sigint:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([(logl()[-1] * 1000).astype("H").astype("U4")])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([(loga()[-1] * 1000).astype("H").astype("U3")])
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
def Ret(i, h=0, k=0, v=0, **_):
    h = h or (i / 4).__ceil__()
    k = k or (i / h).__ceil__()
    v = v or k
    return Res(
        Add(
            Seq(
                Dot(
                    Seq(Lin(i, h * k, True), Unflatten(-1, (h, 1, k))),
                    Seq(
                        Mul(
                            Seq(Lin(i, h * k, True), Unflatten(-1, (h, 1, k))),
                            Seq(Lin(i, h * v, True), Unflatten(-1, (h, v, 1))),
                        ),
                        Decay((h, 1, k), (h, v, 1), d=-4, shift=1, **_),
                    ),
                ),
                Flatten(-3),
                Lin(h * v, i),
            ),
            Lin(i, i),
        ),
        *Ba(i),
        Lin(i, i),
    )


m = 64
m = Seq(
    Embedding(68, m, scale_grad_by_freq=True),
    Ret(m),
    Ret(m),
    Ret(m),
    Ret(m),
    Linear(m, 68),
).to("cuda", non_blocking=True)
print(m)
m = torch.compile(torch.jit.trace(m, dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-4)
s = torch.cuda.amp.GradScaler()
logl = Log()
loga = Log()
for i, x in enumerate(dat):
    o.zero_grad()
    with torch.autocast("cuda", torch.bfloat16):
        y = m(x[..., :-1])
        x = x[..., 1:]
        l = functional.cross_entropy(y.mT, x)
    s.scale(l).backward()
    s.step(o)
    s.update()
    logl(l.data)
    loga((y.argmax(-1) != x).sum() / x.numel())
    if not (i + 1) % 100:
        plt.subplot(211).plot(logl())
        plt.yscale("log")
        plt.legend([logl()[-1]])
        plt.subplot(212).plot(loga())
        plt.yscale("log")
        plt.legend([loga()[-1]])
        clear_output(True)
        plt.show()